# Venta - 01 EDA

Exploracion inicial de los datos de venta inmobiliaria en Moscu.

Objetivo: entender estructura, calidad, variables disponibles y posibles relaciones con el precio de venta.

## 0. Configuracion del entorno

Ejecutar esta celda solo la primera vez que se configura el entorno o cuando se modifiquen las dependencias en `requirements.txt`.

In [ ]:
%pip install -r ../requirements.txt

## Imports

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

DATA_DIR = Path('..') / 'Data'

pd.set_option('display.max_columns', 100)
sns.set_theme(style='whitegrid')

## Carga de datos

In [ ]:
secondary = pd.read_csv(DATA_DIR / 'secondary_market.csv')
new_builds = pd.read_csv(DATA_DIR / 'new_builds.csv')
district_prices = pd.read_csv(DATA_DIR / 'district_prices_monthly.csv')
metro = pd.read_csv(DATA_DIR / 'metro_stations.csv')

datasets = {
    'secondary_market': secondary,
    'new_builds': new_builds,
    'district_prices_monthly': district_prices,
    'metro_stations': metro,
}

for name, df in datasets.items():
    print(f'{name}: {df.shape[0]:,} filas x {df.shape[1]:,} columnas')

In [ ]:
display(secondary.head())
display(new_builds.head())
display(district_prices.head())
display(metro.head())

## Estructura, nulos y duplicados

In [ ]:
schema_summary = []

for name, df in datasets.items():
    tmp = pd.DataFrame({
        'dataset': name,
        'columna': df.columns,
        'tipo': df.dtypes.astype(str).values,
        'nulos': df.isna().sum().values,
        'pct_nulos': df.isna().mean().round(4).values,
        'unicos': df.nunique(dropna=True).values,
    })
    schema_summary.append(tmp)

schema_summary = pd.concat(schema_summary, ignore_index=True)
display(schema_summary)

In [ ]:
for name, df in datasets.items():
    print(f'{name}: {df.duplicated().sum():,} filas duplicadas')

## Variables numericas

In [ ]:
display(secondary.describe(include=[np.number]).T)
display(new_builds.describe(include=[np.number]).T)

In [ ]:
for name, df in {'secondary_market': secondary, 'new_builds': new_builds}.items():
    numeric_cols = df.select_dtypes(include=np.number).columns[:8]
    if len(numeric_cols) == 0:
        continue
    df[numeric_cols].hist(figsize=(14, 9), bins=30)
    plt.suptitle(f'Distribuciones numericas - {name}', y=1.02)
    plt.tight_layout()
    plt.show()

## Variables categoricas

In [ ]:
for name, df in {'secondary_market': secondary, 'new_builds': new_builds}.items():
    print(f'\n{name}')
    for col in df.select_dtypes(exclude=np.number).columns[:8]:
        print(f'\n{col}')
        display(df[col].value_counts(dropna=False).head(10).to_frame('cantidad'))

## Variables candidatas de precio

In [ ]:
tokens_precio = ['price', 'cost', 'value']

for name, df in {'secondary_market': secondary, 'new_builds': new_builds}.items():
    price_cols = [col for col in df.columns if any(token in col.lower() for token in tokens_precio)]
    print(name, price_cols)
    if price_cols:
        display(df[price_cols].describe().T)

In [ ]:
numeric_cols = secondary.select_dtypes(include=np.number).columns

if len(numeric_cols) >= 2:
    corr = secondary[numeric_cols].corr(numeric_only=True)
    plt.figure(figsize=(12, 9))
    sns.heatmap(corr, cmap='coolwarm', center=0)
    plt.title('Correlacion numerica - secondary_market')
    plt.show()

## Notas para continuar

- Identificar columna objetivo de precio de venta.
- Revisar outliers en precio, superficie, habitaciones y distancia a metro/centro si existen.
- Definir llaves utiles para unir informacion de distritos, precios mensuales o estaciones de metro.
- Documentar variables que requieren limpieza antes de modelado.